# Debug: Informative Image-Only — Compare Our Code vs Reference Notebook

Runs the **exact same test** using two approaches:
1. **Our code** (`scripts/zeroshot_llama.py`)
2. **Reference code** (copied from `temp/llama-3-fewshot-Informative.ipynb`)

Both use the same model, same data, same prompt. This isolates whether the difference is in the inference code or the data.

## Setup — Load Model Once

In [1]:
import torch
import pandas as pd
import os, re, time
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f"Model loaded on {model.device}")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded on cuda:0


## Load Test Data (shared by both approaches)

In [2]:
# Load from our preprocessed TSV (image_only)
data_path = os.path.abspath("../data/CrisisMMD/tasks/informative/image_only/test.tsv")
df = pd.read_csv(data_path, sep="\t")
print(f"Loaded {len(df)} samples")
print(f"Columns: {list(df.columns)}")
print(f"Label distribution:\n{df['class_label'].value_counts()}")
print(f"\nFirst row:")
print(df.iloc[0])

Loaded 1534 samples
Columns: ['image_id', 'image_path', 'class_label']
Label distribution:
class_label
informative        1030
not_informative     504
Name: count, dtype: int64

First row:
image_id                                    878185882431389696_0
image_path     D:\Workspace\Cotrain_CrisisMMD\data\CrisisMMD\...
class_label                                      not_informative
Name: 0, dtype: str


## Approach A: 100% verbatim reference code

Everything below is copied directly from `temp/llama-3-fewshot-Informative.ipynb` — including the `test_model` function which loads its own fresh model instance.

In [3]:
## --- 100% verbatim code from temp/llama-3-fewshot-Informative.ipynb ---

INFO_IMAGE_ONLY_PROMPT = """Does the image give useful information that could help during a crisis?
Respond with '1' if this image provides any information or details about a crisis, and '0' if it does not.

Instructions: 
  - You should prioritize identifying texts that provide relevant details, even if they are brief or incomplete.
  - Avoid being overly restrictive. If the text has any relevant crisis-related information, response with '1'.
  - When the meaning of the image is unclear, response with '0'.
  - Do not output any extra text.
"""

def detect_informative_label(text):
    if re.search(r"\bnot (provide |give |contain |related |relevant )?any information\b", text, re.IGNORECASE) or re.search(r"\b0[.!]?\b", text):
        print("Analyze successfully! This entry is not informative!")
        return 0
    elif re.search(r"\b(provides |provide |gives |give )(useful |some |relevant )?information\b", text, re.IGNORECASE)  or re.search(r"\b1[.!]?\b", text):
        print("Analyze successfully! This entry is informative!")
        return 1
    print("Analyze completed! No matching label found! Assigned as not informative by default!") 
    return 0

def load_and_process_image(image_path):
    """Loads and processes an image (returns a PIL Image) for the model."""
    assert os.path.isfile(image_path), f"File not found: {image_path}"
    return Image.open(image_path).convert("RGB")

def construct_message_template(few_shot_examples, prompt, test_text=None, test_image=None, use_texts=True, use_images=True, few_shot_num=0):
    messages = []

    def ordinal(n):
        if 10 <= n % 100 <= 20:
            suffix = "th"
        else:
            suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
        return f"{n}{suffix}"

    user_content = []
    if few_shot_num > 0 and few_shot_examples:          
        for index, example in enumerate(few_shot_examples):
            txt_prompt = "Below is an example for your reference:\n"
            if use_texts and use_images:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the image, and here is the Text: {example['text']}\nThe category of this image and Text is: "
            elif use_texts:
                txt_prompt = f"Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text is: "
            else:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image. The category of this sample image is: "
            txt_prompt += str(example['label'])
            user_content.append({"type": "text", "text": txt_prompt})    

    txt_prompt = ""

    if use_texts and use_images:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the given image, and here is the given Text: {test_text}.\nThe category of this image and Text is: "
    elif use_texts:
        txt_prompt = f"{prompt}\nHere is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test Text is: "
    else:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image that you need to classify. The category of this test image is: "

    user_content.append({"type": "text", "text": txt_prompt})    
    messages.append({"role": "user", "content": user_content})             

    return messages

def classify_raw_text(raw_text, task="Informative"):
    parts = raw_text.split("assistant")
    
    if len(parts) > 1:
        final_answer = parts[-1].strip()
        if task=="Informative":            
            if final_answer.startswith("1"):
                return 1
            elif final_answer.startswith("0"):
                return 0
            else:
                print(f"Unclear model output: {final_answer}! Trying to analyze the output again...")
                return detect_informative_label(final_answer)
    else:
        print(f"Could not parse output: {raw_text}")
        return None

MSG_PRINTED = False

def predict(model, processor, prompt, task="Informative", 
            use_texts=True, use_images=True, 
            text=None, image_path=None, 
            few_shot_examples=None, few_shot_num=0):
    global MSG_PRINTED

    test_image = load_and_process_image(image_path) if image_path else None
    all_images = []

    if use_images:
        if few_shot_num > 0 and few_shot_examples:
            for example in few_shot_examples:
                all_images.append(example["image"])       
            if test_image:
                all_images.append(test_image)
        elif test_image:
            all_images = test_image
        else:
            all_images = None

    messages = construct_message_template(
        few_shot_examples=few_shot_examples,
        prompt=prompt,
        test_text=text,
        test_image=test_image,
        use_texts=use_texts,
        use_images=use_images,
        few_shot_num=few_shot_num
    )

    if not MSG_PRINTED:
        def json_serializable(obj):
            if isinstance(obj, Image.Image):
                return f"<PIL.Image mode={obj.mode} size={obj.size}>"
            raise TypeError(f"Type {type(obj)} is not JSON serializable")
        print("\n\n", '-'*80)
        print(json.dumps(messages, indent=2, ensure_ascii=False, default=json_serializable))
        print('-'*80, "\n\n")
        MSG_PRINTED = True

    formatted_input = processor.apply_chat_template(messages, add_generation_prompt=True)

    if use_images:
        inputs = processor(
            text=formatted_input,
            images=all_images,
            return_tensors="pt",
            truncation=False,
        ).to(model.device)
    else:
        inputs = processor(
            text=formatted_input,            
            return_tensors="pt",
            truncation=False,
        ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            min_new_tokens=1,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    raw_text = processor.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    
    return classify_raw_text(raw_text, task)

def update_remaining_time(start_time, completed, total, label=""):
    elapsed = time.time() - start_time
    rate = completed / elapsed if elapsed > 0 else 0
    remaining = (total - completed) / rate if rate > 0 else 0
    print(f"  [{label}] {completed}/{total} done, ~{remaining:.0f}s remaining")

print("All reference functions loaded")

All reference functions loaded


In [5]:
## --- Run Approach A: Exact reference notebook flow using the JSON dataset ---
import json

# Load test data from the SAME JSON the reference notebook used
test_data = pd.read_json("../temp/info_image_only_test.json", lines=False, encoding="utf-8")

# Fix image paths: JSON has relative paths like "data_image/..." 
# Need to prepend the base path to make them absolute
IMAGE_BASE = os.path.abspath("../data/CrisisMMD")
test_data["image_path"] = test_data["image_path"].apply(lambda p: os.path.join(IMAGE_BASE, p))
print(f"Loaded {len(test_data)} samples from reference JSON")
print(f"Sample path: {test_data.iloc[0]['image_path']}")

MSG_PRINTED = False
results_ref = []
y_true_ref = []
y_pred_ref = []
start_time = time.time()

for index, row in test_data.iterrows():
    image_path = row["image_path"]
    y_true = row["label"]
    text = row["text"]

    y_pred = predict(
        model=model,
        processor=processor,
        prompt=INFO_IMAGE_ONLY_PROMPT,
        task="Informative",
        use_texts=False,
        use_images=True,
        text=text,
        image_path=image_path,
        few_shot_examples=None,
        few_shot_num=0
    )

    if y_pred is None:
        print(f"\nBad result at index {index}! Skipping.\n")
        continue

    results_ref.append({"image_path": image_path, "y_true": y_true, "y_pred": y_pred})
    y_true_ref.append(y_true)
    y_pred_ref.append(y_pred)

    if (index + 1) % 200 == 0:
        update_remaining_time(start_time, index + 1, len(test_data), f"{index+1}/{len(test_data)}")
        torch.cuda.empty_cache()

# Calculate metrics exactly as reference notebook does
accuracy = accuracy_score(y_true_ref, y_pred_ref)
precision, recall, f1, _ = precision_recall_fscore_support(y_true_ref, y_pred_ref, average="weighted")

print(f"\n=== Approach A: 100% Reference Notebook Code ===")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  (Reference reported: Accuracy=0.8605, F1=0.8534)")

Loaded 1534 samples from reference JSON
Sample path: D:\Workspace\Cotrain_CrisisMMD\data\CrisisMMD\data_image/srilanka_floods/23_6_2017/878185882431389696_0.jpg


 --------------------------------------------------------------------------------
[
  {
    "role": "user",
    "content": [
      {
        "type": "image",
        "image": "<PIL.Image mode=RGB size=(1200, 799)>"
      },
      {
        "type": "text",
        "text": "Does the image give useful information that could help during a crisis?\nRespond with '1' if this image provides any information or details about a crisis, and '0' if it does not.\n\nInstructions: \n  - You should prioritize identifying texts that provide relevant details, even if they are brief or incomplete.\n  - Avoid being overly restrictive. If the text has any relevant crisis-related information, response with '1'.\n  - When the meaning of the image is unclear, response with '0'.\n  - Do not output any extra text.\n\nAbove is the test image that you ne

D:\Workspace\Cotrain_CrisisMMD\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\Workspace\Cotrain_CrisisMMD\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\Workspace\Cotrain_CrisisMMD\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Unclear model output: The image is a screenshot of a music player app on a mobile device, displaying the song "Party! Trying to analyze the output again...
Analyze completed! No matching label found! Assigned as not informative by default!
Unclear model output: The image shows a weather radar map with a red box around a specific area, indicating a severe weather! Trying to analyze the output again...
Analyze completed! No matching label found! Assigned as not informative by default!
  [200/1534] 200/1534 done, ~955s remaining
  [400/1534] 400/1534 done, ~817s remaining
Unclear model output: The image shows a woman doing yoga on the beach, which is not related to a crisis. Therefore! Trying to analyze the output again...
Analyze completed! No matching label found! Assigned as not informative by default!
  [600/1534] 600/1534 done, ~679s remaining
  [800/1534] 800/1534 done, ~536s remaining
  [1000/1534] 1000/1534 done, ~390s remaining
  [1200/1534] 1200/1534 done, ~244s remaining
  [140

## Approach B: Our Code

Uses `scripts/zeroshot_llama.py` on the same data.

In [3]:
import sys
sys.path.insert(0, os.path.abspath(".."))

from scripts.zeroshot_llama import run_zeroshot

DATA_ROOT = os.path.abspath("../data")
OUTPUT_DIR = os.path.abspath("../results/zeroshot_debug")

preds_ours, metrics_ours = run_zeroshot(
    task="informative", modality="image_only", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)

print(f"\n=== Approach B: Our Code ===")
print(f"  Accuracy:    {metrics_ours['accuracy']:.4f}")
print(f"  Weighted F1: {metrics_ours['weighted_f1']:.4f}")
print(f"  Macro F1:    {metrics_ours['macro_f1']:.4f}")

Loaded 1534 samples from D:\Workspace\Cotrain_CrisisMMD\data\CrisisMMD\tasks\informative\image_only\test.tsv
  [50/1534] 36s elapsed, ~1062s remaining, 1.4 samples/s
  [100/1534] 72s elapsed, ~1025s remaining, 1.4 samples/s
  [150/1534] 109s elapsed, ~1007s remaining, 1.4 samples/s
  [200/1534] 145s elapsed, ~970s remaining, 1.4 samples/s
  [250/1534] 182s elapsed, ~936s remaining, 1.4 samples/s
  [300/1534] 219s elapsed, ~901s remaining, 1.4 samples/s
  [350/1534] 255s elapsed, ~863s remaining, 1.4 samples/s
  [400/1534] 290s elapsed, ~822s remaining, 1.4 samples/s
  [450/1534] 327s elapsed, ~787s remaining, 1.4 samples/s
  [500/1534] 364s elapsed, ~753s remaining, 1.4 samples/s
  [550/1534] 401s elapsed, ~717s remaining, 1.4 samples/s
  [600/1534] 437s elapsed, ~681s remaining, 1.4 samples/s
  [650/1534] 474s elapsed, ~645s remaining, 1.4 samples/s
  [700/1534] 511s elapsed, ~609s remaining, 1.4 samples/s
  [750/1534] 548s elapsed, ~573s remaining, 1.4 samples/s
  [800/1534] 585s ela

## Compare: Side by Side + Per-Sample Diff

In [ ]:
print("=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"{'':30s} {'Ref':>10s} {'Ours':>10s}")
print(f"{'Accuracy':30s} {acc_ref:>10.4f} {metrics_ours['accuracy']:>10.4f}")
print(f"{'Weighted F1':30s} {f1_ref:>10.4f} {metrics_ours['weighted_f1']:>10.4f}")
print(f"{'Macro F1':30s} {mf1_ref:>10.4f} {metrics_ours['macro_f1']:>10.4f}")

# Per-sample comparison
label_map_inv = {1: "informative", 0: "not_informative"}
y_pred_ours = [p["predicted_label"] for p in preds_ours]
y_pred_ref_str = [label_map_inv.get(p, "not_informative") for p in y_pred_ref]

agree = sum(a == b for a, b in zip(y_pred_ref_str, y_pred_ours))
print(f"\nPer-sample agreement: {agree}/{len(df)} ({agree/len(df)*100:.1f}%)")

# Show first 10 disagreements
disagree_idx = [i for i, (a, b) in enumerate(zip(y_pred_ref_str, y_pred_ours)) if a != b]
print(f"Disagreements: {len(disagree_idx)}")
if disagree_idx:
    print(f"\nFirst 10 disagreements:")
    for idx in disagree_idx[:10]:
        print(f"  [{idx}] gold={df.iloc[idx]['class_label']}, ref={y_pred_ref_str[idx]}, ours={y_pred_ours[idx]}")

## Cleanup

In [ ]:
import gc
del model, processor
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")